In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Caminho da Camada Bronze nos Volumes do Databricks
BRONZE_PATH = "/Volumes/workspace/olist_bronze/parquet"

# Carregando as 9 tabelas brutas diretamente da Bronze
df_customers = spark.read.parquet(f"{BRONZE_PATH}/olist_customers_dataset")
df_orders = spark.read.parquet(f"{BRONZE_PATH}/olist_orders_dataset")
df_items = spark.read.parquet(f"{BRONZE_PATH}/olist_order_items_dataset")
df_payments = spark.read.parquet(f"{BRONZE_PATH}/olist_order_payments_dataset")
df_reviews = spark.read.parquet(f"{BRONZE_PATH}/olist_order_reviews_dataset")
df_products = spark.read.parquet(f"{BRONZE_PATH}/olist_products_dataset")
df_sellers = spark.read.parquet(f"{BRONZE_PATH}/olist_sellers_dataset")
df_geolocation = spark.read.parquet(f"{BRONZE_PATH}/olist_geolocation_dataset")
df_translation = spark.read.parquet(f"{BRONZE_PATH}/product_category_name_translation")



In [0]:
# ==========================================
# 1. ANÁLISE DA TAXA DE ATRASO GERAL
# ==========================================
# Identificando pedidos entregues em que a data real de entrega foi maior que a estimada
df_orders_delay = df_orders.withColumn(
    "is_delayed",
    F.when(
        (F.col("order_status") == "delivered") & 
        (F.col("order_delivered_customer_date").isNotNull()) & 
        (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")), 1
    ).otherwise(0)
)

print("--- 1. TAXA DE ATRASO GERAL NA BASE ---")
taxa_atraso_df = df_orders_delay.filter(F.col("order_status") == "delivered").groupBy().agg(
    F.count("*").alias("total_entregues"),
    F.sum("is_delayed").alias("total_atrasados"),
    (F.sum("is_delayed") / F.count("*") * 100).alias("taxa_atraso_pct")
)
display(taxa_atraso_df)


# ==========================================
# 2. IMPACTO DO ATRASO NA SATISFAÇÃO (REVIEWS)
# ==========================================
# Cruzando pedidos com as avaliações dos clientes
df_orders_reviews = df_orders_delay.join(df_reviews, "order_id", "inner")

print("--- 2. IMPACTO DO ATRASO NA NOTA MÉDIA E REVIEWS RUINS (1-2) ---")
impacto_satisfacao_df = df_orders_reviews.filter(F.col("order_status") == "delivered").groupBy("is_delayed").agg(
    F.count("*").alias("qtd_pedidos"),
    F.round(F.avg("review_score"), 2).alias("media_review_score"),
    F.round(
        (F.sum(F.when(F.col("review_score").isin(1, 2), 1).otherwise(0)) / F.count("*")) * 100, 2
    ).alias("pct_notas_1_2")
)
display(impacto_satisfacao_df)

In [0]:
# ==========================================
# 3. PROPORÇÃO DO FRETE POR FAIXA DE PREÇO
# ==========================================
# Calculando o peso do frete sobre o preço do produto nas faixas de preço
df_items_enriched = df_items.withColumn(
    "faixa_preco",
    F.when(F.col("price") <= 50, "1. Até R$ 50")
     .when((F.col("price") > 50) & (F.col("price") <= 200), "2. R$ 50 - R$ 200")
     .when((F.col("price") > 200) & (F.col("price") <= 500), "3. R$ 200 - R$ 500")
     .otherwise("4. Acima de R$ 500")
).withColumn(
    "frete_pct_valor", (F.col("freight_value") / F.col("price")) * 100
)

print("--- 3. PESO DO FRETE NO TICKET DO PRODUTO POR FAIXA ---")
frete_faixa_df = df_items_enriched.groupBy("faixa_preco").agg(
    F.count("*").alias("total_itens"),
    F.round(F.avg("price"), 2).alias("preco_medio"),
    F.round(F.avg("freight_value"), 2).alias("frete_medio"),
    F.round(F.avg("frete_pct_valor"), 2).alias("media_frete_pct_do_preco")
).orderBy("faixa_preco")

display(frete_faixa_df)


